# 📖 Lab 5: Scaling the View Path (Deep Dive)

**Non-functional requirements:**
- *The system should be scalable and handle high throughput (10M users, one event)*
- *The system is read-heavy (100:1 read-to-write ratio)*
- *Prioritize availability for viewing events*

When Taylor Swift tickets go on sale, **millions of users** refresh the same event page simultaneously. Every refresh hits `GET /events/:eventId` — the same query, returning the same event + venue + performer data. Without optimization, every request hits PostgreSQL directly, and the database melts.

## 🏗️ Architecture — Before (Lab 1)

```
┌────────┐       ┌─────────────┐       ┌───────────────┐       ┌──────────────┐
│ Client │──────>│ API Gateway │──────>│ Event Service  │──SQL─>│  PostgreSQL  │
└────────┘       └─────────────┘       └───────────────┘       └──────────────┘
                                        Every request
                                        hits the DB! 💀
```

## 🏗️ Architecture — After (Caching + Horizontal Scaling)

```
┌────────┐       ┌─────────────┐       ┌───────────────────────────────────────┐
│        │       │             │       │  Event Service (N instances)           │
│        │       │    Load     │       │  ┌─────────┐ ┌─────────┐ ┌─────────┐ │
│ Client │──────>│  Balancer   │──────>│  │ Inst. 1 │ │ Inst. 2 │ │ Inst. N │ │
│        │       │  (RR / LC)  │       │  └────┬────┘ └────┬────┘ └────┬────┘ │
└────────┘       └─────────────┘       └───────┼──────────┼──────────┼───────┘
                                               │          │          │
                                               v          v          v
                                        ┌──────────────┐
                                        │    Redis     │  read-through
                                        │  Event Cache │  cache
                                        │              │
                                        │ event:{id}   │  TTL: 5 min (static)
                                        │ tickets:{id} │  TTL: 10s (dynamic)
                                        └──────┬───────┘
                                               │ cache MISS only
                                               v
                                        ┌──────────────┐
                                        │  PostgreSQL  │
                                        │ (source of   │
                                        │   truth)     │
                                        └──────────────┘
```

## Learning Objectives

- Measure the problem: how many DB queries per second before it breaks
- Implement a read-through Redis cache for event data
- Compare response times: DB-only vs cached
- Understand cache invalidation strategies (TTL, write-through)
- See why event data is a perfect caching candidate (high reads, low writes)
- Simulate horizontal scaling with concurrent "service instances"

## 🛠️ Setup

Make sure PostgreSQL and Redis are running:

```bash
cd system-designs/ticketmaster
docker-compose up -d
```

Select the **"Ticketmaster (Python)"** kernel.

In [1]:
import psycopg2
import psycopg2.extras
import redis
import json
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

DB_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "user": "demo",
    "password": "demo",
    "database": "ticketmaster",
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

redis_client = redis.Redis(host="localhost", port=6380, decode_responses=True)

# Flush any leftover cache from previous runs
redis_client.flushdb()

conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM events")
print(f"✅ PostgreSQL: {cur.fetchone()[0]} events")
cur.close()
conn.close()
print(f"✅ Redis: {'connected' if redis_client.ping() else 'FAILED'}")

✅ PostgreSQL: 5 events
✅ Redis: connected


## 📊 The Problem: Every Request Hits the Database

In Lab 1, our `GET /events/:eventId` handler runs two SQL queries on every request:
1. A 3-table JOIN (events + venues + performers)
2. A ticket query (all tickets for the event)

When 10,000 users refresh the same event page every second, that's **20,000 SQL queries/second** — for data that hasn't changed since the event was created.

Let's measure the baseline.

In [2]:
def get_event_from_db(event_id: int) -> dict:
    """Original Lab 1 approach: every call hits PostgreSQL."""
    conn = get_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cur.execute("""
        SELECT 
            e.id AS event_id, e.name AS event_name, e.description AS event_description,
            e.event_type, e.event_date, e.status AS event_status,
            v.id AS venue_id, v.name AS venue_name, v.address, v.city, v.state, v.country,
            v.capacity, v.seat_map,
            p.id AS performer_id, p.name AS performer_name, p.description AS performer_description,
            p.genre, p.image_url
        FROM events e
        JOIN venues v ON e.venue_id = v.id
        JOIN performers p ON e.performer_id = p.id
        WHERE e.id = %s
    """, (event_id,))
    row = cur.fetchone()

    cur.execute("""
        SELECT id, section, row_label, seat_number, price, status
        FROM tickets WHERE event_id = %s
        ORDER BY section, row_label, seat_number
    """, (event_id,))
    tickets = cur.fetchall()

    cur.close()
    conn.close()

    return {"event": dict(row), "tickets": [dict(t) for t in tickets]}


def load_test(func, event_id: int, num_requests: int, concurrency: int) -> dict:
    """Simulate concurrent users hitting the same endpoint."""
    latencies = []

    def single_request():
        start = time.time()
        func(event_id)
        return (time.time() - start) * 1000  # ms

    with ThreadPoolExecutor(max_workers=concurrency) as executor:
        futures = [executor.submit(single_request) for _ in range(num_requests)]
        for f in as_completed(futures):
            latencies.append(f.result())

    latencies.sort()
    return {
        "total_requests": num_requests,
        "concurrency": concurrency,
        "avg_ms": round(sum(latencies) / len(latencies), 2),
        "p50_ms": round(latencies[len(latencies) // 2], 2),
        "p95_ms": round(latencies[int(len(latencies) * 0.95)], 2),
        "p99_ms": round(latencies[int(len(latencies) * 0.99)], 2),
        "max_ms": round(max(latencies), 2),
    }


# Baseline: all requests hit PostgreSQL
print("📊 Baseline: Every request hits PostgreSQL directly\n")
print(f"{'Concurrency':<15} {'Avg (ms)':<12} {'P50':<10} {'P95':<10} {'P99':<10} {'Max'}")
print("-" * 65)

for concurrency in [1, 5, 10, 25, 50]:
    stats = load_test(get_event_from_db, event_id=1, num_requests=100, concurrency=concurrency)
    print(f"{concurrency:<15} {stats['avg_ms']:<12} {stats['p50_ms']:<10} {stats['p95_ms']:<10} {stats['p99_ms']:<10} {stats['max_ms']}")

print(f"\n⚠️  Notice: latency increases with concurrency.")
print(f"   Each request opens a DB connection, runs 2 queries, and closes it.")
print(f"   At 10M concurrent users, this would overwhelm any single database.")

📊 Baseline: Every request hits PostgreSQL directly

Concurrency     Avg (ms)     P50        P95        P99        Max
-----------------------------------------------------------------
1               13.95        12.63      21.52      54.97      54.97
5               16.52        16.68      19.59      21.31      21.31
10              33.34        28.13      75.51      79.3       79.3
25              73.18        74.17      101.67     116.16     116.16
50              168.35       170.46     248.66     279.36     279.36

⚠️  Notice: latency increases with concurrency.
   Each request opens a DB connection, runs 2 queries, and closes it.
   At 10M concurrent users, this would overwhelm any single database.


## 🚀 Solution: Read-Through Redis Cache

The insight: event data (name, date, venue, performer) **rarely changes**. There's no reason to query PostgreSQL for the same data millions of times.

### What to cache

| Data | Update Frequency | Cache TTL |
|------|-------------------|-----------|
| Event details (name, date, type) | Almost never | 5–10 minutes |
| Venue (name, address, capacity, seat map) | Never | 1 hour |
| Performer (name, bio, genre) | Rarely | 30 minutes |
| Ticket statuses (available/sold) | Frequently (on every booking) | 5–15 seconds |

### Read-through cache strategy

```
Request comes in for event 123
  → Check Redis: GET event:123
    → Cache HIT?  Return cached data immediately ✅
    → Cache MISS? Query PostgreSQL → Store in Redis with TTL → Return data
```

This means the first request is slow (DB hit), but subsequent requests are **100x faster** (Redis only).

In [3]:
# Cache TTLs (in seconds)
EVENT_CACHE_TTL = 300       # 5 minutes — event details rarely change
TICKETS_CACHE_TTL = 10      # 10 seconds — ticket statuses change on bookings

def get_event_cached(event_id: int) -> dict:
    """
    Read-through cache: check Redis first, fall back to PostgreSQL on miss.
    Caches event+venue+performer separately from tickets (different TTLs).
    """
    cache_key_event = f"event:{event_id}"
    cache_key_tickets = f"event:{event_id}:tickets"

    # Try cache first
    cached_event = redis_client.get(cache_key_event)
    cached_tickets = redis_client.get(cache_key_tickets)

    if cached_event and cached_tickets:
        # CACHE HIT — no DB query needed
        return {
            "event": json.loads(cached_event),
            "tickets": json.loads(cached_tickets),
            "source": "cache",
        }

    # CACHE MISS — query DB and populate cache
    conn = get_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    if not cached_event:
        cur.execute("""
            SELECT 
                e.id AS event_id, e.name AS event_name, e.description AS event_description,
                e.event_type, e.event_date, e.status AS event_status,
                v.id AS venue_id, v.name AS venue_name, v.address, v.city, v.state, v.country,
                v.capacity, v.seat_map,
                p.id AS performer_id, p.name AS performer_name,
                p.description AS performer_description, p.genre, p.image_url
            FROM events e
            JOIN venues v ON e.venue_id = v.id
            JOIN performers p ON e.performer_id = p.id
            WHERE e.id = %s
        """, (event_id,))
        row = cur.fetchone()
        # Convert datetime to string for JSON serialization
        event_data = dict(row)
        event_data["event_date"] = str(event_data["event_date"])
        redis_client.setex(cache_key_event, EVENT_CACHE_TTL, json.dumps(event_data))
    else:
        event_data = json.loads(cached_event)

    if not cached_tickets:
        cur.execute("""
            SELECT id, section, row_label, seat_number, price, status
            FROM tickets WHERE event_id = %s
            ORDER BY section, row_label, seat_number
        """, (event_id,))
        tickets_data = [dict(t) for t in cur.fetchall()]
        # Convert Decimal to float for JSON
        for t in tickets_data:
            t["price"] = float(t["price"])
        redis_client.setex(cache_key_tickets, TICKETS_CACHE_TTL, json.dumps(tickets_data))
    else:
        tickets_data = json.loads(cached_tickets)

    cur.close()
    conn.close()

    return {
        "event": event_data,
        "tickets": tickets_data,
        "source": "db",
    }

print("✅ get_event_cached() defined — read-through cache with split TTLs.")

✅ get_event_cached() defined — read-through cache with split TTLs.


## 🧪 Test: Cache Miss → Cache Hit

Let's see the difference between the first request (cache miss, hits DB) and subsequent requests (cache hit, Redis only).

In [4]:
# Clear cache first
redis_client.delete("event:1", "event:1:tickets")

# Request 1: Cache MISS
start = time.time()
result1 = get_event_cached(event_id=1)
miss_ms = (time.time() - start) * 1000

print(f"📥 Request 1 (cache MISS):")
print(f"   Source: {result1['source']}")
print(f"   Time: {miss_ms:.2f}ms")
print(f"   Event: {result1['event']['event_name']}")
print(f"   Tickets: {len(result1['tickets'])}")

# Request 2: Cache HIT
start = time.time()
result2 = get_event_cached(event_id=1)
hit_ms = (time.time() - start) * 1000

print(f"\n📦 Request 2 (cache HIT):")
print(f"   Source: {result2['source']}")
print(f"   Time: {hit_ms:.2f}ms")

speedup = miss_ms / hit_ms if hit_ms > 0 else float("inf")
print(f"\n🚀 Speedup: {speedup:.0f}x faster from cache!")
print(f"   Cache TTL: event data = {EVENT_CACHE_TTL}s, tickets = {TICKETS_CACHE_TTL}s")
print(f"   Redis key: event:1 TTL = {redis_client.ttl('event:1')}s")

📥 Request 1 (cache MISS):
   Source: db
   Time: 23.39ms
   Event: The Eras Tour - NYC
   Tickets: 115

📦 Request 2 (cache HIT):
   Source: cache
   Time: 1.04ms

🚀 Speedup: 22x faster from cache!
   Cache TTL: event data = 300s, tickets = 10s
   Redis key: event:1 TTL = 300s


## 📈 Load Test: DB-Only vs Cached Under Concurrent Load

Now let's compare both approaches under simulated concurrent traffic — as if hundreds of users are refreshing the event page at the same time.

In [5]:
# Make sure cache is warm
get_event_cached(event_id=1)

NUM_REQUESTS = 200
CONCURRENCY = 25

print(f"📊 Load Test: {NUM_REQUESTS} requests, {CONCURRENCY} concurrent\n")

# Test 1: DB-only
print("────────────────── DB-ONLY ──────────────────")
db_stats = load_test(get_event_from_db, event_id=1, num_requests=NUM_REQUESTS, concurrency=CONCURRENCY)
print(f"  Avg: {db_stats['avg_ms']}ms | P50: {db_stats['p50_ms']}ms | P95: {db_stats['p95_ms']}ms | P99: {db_stats['p99_ms']}ms | Max: {db_stats['max_ms']}ms")

# Test 2: Cached
print("\n────────────────── CACHED ──────────────────")
cache_stats = load_test(get_event_cached, event_id=1, num_requests=NUM_REQUESTS, concurrency=CONCURRENCY)
print(f"  Avg: {cache_stats['avg_ms']}ms | P50: {cache_stats['p50_ms']}ms | P95: {cache_stats['p95_ms']}ms | P99: {cache_stats['p99_ms']}ms | Max: {cache_stats['max_ms']}ms")

# Comparison
print(f"\n{'─'*55}")
print(f"{'Metric':<12} {'DB-Only':<15} {'Cached':<15} {'Speedup'}")
print(f"{'─'*55}")
for metric in ["avg_ms", "p50_ms", "p95_ms", "p99_ms"]:
    label = metric.replace("_ms", "").upper()
    db_val = db_stats[metric]
    cache_val = cache_stats[metric]
    speedup = db_val / cache_val if cache_val > 0 else float("inf")
    print(f"{label:<12} {db_val:<15} {cache_val:<15} {speedup:.1f}x")

print(f"\n💡 The cached version serves the same data but barely touches the database.")
print(f"   In production, this means the DB handles booking writes while Redis handles")
print(f"   millions of read requests per second.")

📊 Load Test: 200 requests, 25 concurrent

────────────────── DB-ONLY ──────────────────
  Avg: 70.56ms | P50: 69.7ms | P95: 88.2ms | P99: 113.04ms | Max: 113.46ms

────────────────── CACHED ──────────────────
  Avg: 4.48ms | P50: 3.56ms | P95: 9.58ms | P99: 13.66ms | Max: 14.09ms

───────────────────────────────────────────────────────
Metric       DB-Only         Cached          Speedup
───────────────────────────────────────────────────────
AVG          70.56           4.48            15.7x
P50          69.7            3.56            19.6x
P95          88.2            9.58            9.2x
P99          113.04          13.66           8.3x

💡 The cached version serves the same data but barely touches the database.
   In production, this means the DB handles booking writes while Redis handles
   millions of read requests per second.


## 🔄 Cache Invalidation

The hardest problem in computer science (after naming things). When event data changes, the cache needs to be updated. Here are the strategies:

### Strategy 1: TTL-based expiration (passive)
Cache entries expire after a set time. Simple, but there's a window where stale data is served.

### Strategy 2: Write-through invalidation (active)
When the Booking Service updates a ticket status, it also invalidates the cache. This keeps the cache in sync immediately.

Let's implement write-through invalidation for ticket bookings.

In [ ]:
def invalidate_ticket_cache(event_id: int):
    """
    Write-through invalidation: called after any ticket status change.
    Deletes the tickets cache so the next read gets fresh data from DB.
    We keep the event cache (name, venue, performer) — that hasn't changed.
    """
    redis_client.delete(f"event:{event_id}:tickets")


def book_ticket_with_cache_invalidation(event_id: int, ticket_id: int):
    """Simulates a booking that also invalidates the cache."""
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("UPDATE tickets SET status = 'booked' WHERE id = %s AND status = 'available'", (ticket_id,))
    conn.commit()
    cur.close()
    conn.close()

    # Invalidate the tickets cache — next read will get fresh data
    invalidate_ticket_cache(event_id)


# Demo: cache → book a ticket → cache invalidated → fresh data
print("📦 Step 1: Read from cache (tickets are available)")
result = get_event_cached(event_id=1)
available_before = sum(1 for t in result["tickets"] if t["status"] == "available")
print(f"   Source: {result['source']} | Available tickets: {available_before}")

# Book a ticket
conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT id FROM tickets WHERE event_id = 1 AND status = 'available' ORDER BY id LIMIT 1")
ticket_to_book = cur.fetchone()[0]
cur.close()
conn.close()

print(f"\n🎟️  Step 2: Book ticket {ticket_to_book}")
book_ticket_with_cache_invalidation(event_id=1, ticket_id=ticket_to_book)
print(f"   Ticket booked + cache invalidated")

# Verify: tickets cache is gone, event cache is still there
print(f"\n🔍 Step 3: Cache state after invalidation:")
print(f"   event:1 (details):  {'✅ still cached' if redis_client.exists('event:1') else '❌ gone'}")
print(f"   event:1:tickets:    {'✅ still cached' if redis_client.exists('event:1:tickets') else '🔄 invalidated (will refresh on next read)'}")

# Next read will cache miss on tickets, hit on event details
print(f"\n📥 Step 4: Next read fetches fresh ticket data")
result = get_event_cached(event_id=1)
available_after = sum(1 for t in result["tickets"] if t["status"] == "available")
print(f"   Source: {result['source']} | Available tickets: {available_after}")
print(f"   Ticket count changed: {available_before} → {available_after}")

# Reset the booked ticket
conn = get_connection()
cur = conn.cursor()
cur.execute("UPDATE tickets SET status = 'available' WHERE id = %s", (ticket_to_book,))
conn.commit()
cur.close()
conn.close()
invalidate_ticket_cache(event_id=1)

## 🏗️ Horizontal Scaling

The Event Service is **stateless** — it doesn't store any data locally. Every request just reads from Redis/PostgreSQL and returns a response. This means we can run N copies behind a load balancer.

```
                ┌──> Event Service Instance 1 ──┐
Load Balancer ──┼──> Event Service Instance 2 ──┼──> Redis Cache ──> PostgreSQL
                ├──> Event Service Instance 3 ──┤
                └──> Event Service Instance N ──┘
```

### Load Balancing Algorithms

| Algorithm | How It Works | Best For |
|-----------|-------------|----------|
| **Round Robin** | Rotate through instances sequentially | Equal-capacity servers |
| **Least Connections** | Route to the instance with fewest active connections | Varying request duration |
| **Weighted** | Route more traffic to more powerful instances | Mixed hardware |

Let's simulate multiple service instances handling requests concurrently, all reading from the same cache.

In [ ]:
# Simulate N "service instances" all serving from the same cache
# In production, these would be separate containers/pods behind a load balancer

def simulate_service_instance(instance_id: int, event_id: int, num_requests: int) -> dict:
    """Simulates one service instance handling num_requests."""
    latencies = []
    for _ in range(num_requests):
        start = time.time()
        get_event_cached(event_id)
        latencies.append((time.time() - start) * 1000)
    return {
        "instance": instance_id,
        "requests": num_requests,
        "avg_ms": round(sum(latencies) / len(latencies), 2),
        "total_ms": round(sum(latencies), 2),
    }

# Warm the cache
get_event_cached(event_id=1)

# Simulate 5 instances handling 100 requests each = 500 total
NUM_INSTANCES = 5
REQUESTS_PER_INSTANCE = 100

print(f"🏗️  Simulating {NUM_INSTANCES} service instances, {REQUESTS_PER_INSTANCE} requests each")
print(f"   Total: {NUM_INSTANCES * REQUESTS_PER_INSTANCE} requests\n")

instance_results = []

with ThreadPoolExecutor(max_workers=NUM_INSTANCES) as executor:
    futures = {
        executor.submit(simulate_service_instance, i, 1, REQUESTS_PER_INSTANCE): i
        for i in range(1, NUM_INSTANCES + 1)
    }
    for f in as_completed(futures):
        instance_results.append(f.result())

instance_results.sort(key=lambda x: x["instance"])

print(f"{'Instance':<12} {'Requests':<12} {'Avg (ms)':<12} {'Total (ms)'}")
print("-" * 48)
for r in instance_results:
    print(f"#{r['instance']:<11} {r['requests']:<12} {r['avg_ms']:<12} {r['total_ms']}")

total_requests = sum(r["requests"] for r in instance_results)
overall_avg = sum(r["avg_ms"] for r in instance_results) / len(instance_results)
print(f"\n📊 All {NUM_INSTANCES} instances served {total_requests} requests")
print(f"   Average latency: {overall_avg:.2f}ms per request")
print(f"\n💡 All instances share the same Redis cache.")
print(f"   Adding more instances = handling more users with the same latency.")
print(f"   The database is barely touched — Redis handles the load.")

## 🧹 Cleanup

In [ ]:
redis_client.flushdb()
print("✅ Redis cache cleared.")

## ✅ Summary

### What We Learned

| Layer | What It Does | Impact |
|-------|-------------|--------|
| **Redis Cache** | Stores event data in-memory, avoids DB queries | 10-100x faster response times |
| **Read-through strategy** | Cache miss → DB read → cache populate → return | First request slow, all subsequent fast |
| **Split TTLs** | Long TTL for static data (venue, performer), short for tickets | Balances freshness with performance |
| **Write-through invalidation** | Delete cache on booking → next read gets fresh data | Cache stays consistent with minimal staleness |
| **Horizontal scaling** | N stateless instances behind a load balancer | Linear throughput scaling |

### What to Cache vs What NOT to Cache

| ✅ Cache Aggressively | ❌ Don't Cache (or Short TTL) |
|----------------------|-------------------------------|
| Event name, description, date | Ticket availability (changes on every booking) |
| Venue details, address, seat map | Booking status |
| Performer bio, genre, image | Payment state |
| Search results for popular queries | User-specific data |

### Cache Invalidation Strategies

| Strategy | When to Use |
|----------|-------------|
| **TTL expiration** | Default — simple, works for most data. Set TTL based on how often data changes. |
| **Write-through** | When you need immediate consistency (e.g., ticket status after booking). Invalidate on write. |
| **Event-driven** | DB triggers or message queues notify the cache when data changes. Best for complex invalidation. |

**Pattern reference:** See `patterns/scaling-reads/` for a deeper exploration of read scaling patterns including denormalization, read replicas, and advanced caching strategies.